In [16]:
import numpy as np
import pandas as pd
from datetime import datetime
import requests
import logging
import wikipediaapi
import datefinder
import re

In [18]:
haunted_places = pd.read_csv("../data/haunted_places.tsv", sep='\t')
def get_date_from_text(description):
    dates = list(datefinder.find_dates(description))
    if len(dates) >= 1:
        return dates[0].strftime("%Y/%m/%d")
    date_patterns = [
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\s+\d{1,2},?\s+\d{4}\b',
        r'\b\d{1,2}/\d{1,2}/\d{4}\b',
        r'\b\d{1,2}-\d{1,2}-\d{4}\b',
        r'\b\d{4}\b'
    ]
    
    for pattern in date_patterns:
        match = re.search(pattern, description, re.IGNORECASE)
        if match:
            date_str = match.group(0)
            try:
                if '/' in date_str or '-' in date_str:
                    date_obj = datetime.strptime(date_str, '%m/%d/%Y') if '/' in date_str else datetime.strptime(date_str, '%m-%d-%Y')
                    return date_obj.strftime('%Y/%m/%d')
                else:
                    date_obj = datetime.strptime(date_str, '%B %d, %Y')
                    return date_obj.strftime('%Y/%m/%d')
            except ValueError:
                try:
                    return datetime.strptime(date_str, '%Y').strftime('%Y/01/01')
                except:
                    continue
    return "2025/01/01"
    

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load Data
file_path = '../data/haunted_places.tsv'
df = pd.read_csv(file_path,sep='\t')

# Wikipedia API Setup - Set proper user agent
user_agent = 'DSCI550_HauntedProject/1.0 (https://github.com/maggie-changg/dsci550-assignment1; changmag@usc.edu)'
wiki_wiki = wikipediaapi.Wikipedia(
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI,
    user_agent=user_agent
)

# Cache for Wikipedia results
wikipedia_cache = {}


# Function to search Wikipedia and extract evidence
def get_evidence_from_wikipedia(location, description):
    if location in wikipedia_cache:
        return wikipedia_cache[location]
    
    search_query = location
    try:
        page = wiki_wiki.page(search_query)
        if page.exists():
            text = page.summary[:500]  # Use the first 500 characters of the summary
            # Extract date
            found_date = get_date_from_text(text)
            
            # Cache results
            wikipedia_cache[location] = {
                'date': found_date if found_date else get_date_from_text(description)
            }
            return wikipedia_cache[location]
    except Exception as e:
        logging.warning(f"Wikipedia search failed for {location}: {e}")
    
    # If Wikipedia fails, fallback to description search
    found_date = get_date_from_text(description)
    
    # Cache results
    wikipedia_cache[location] = {
        'date': found_date if found_date else '2025/01/01'
    }
    return wikipedia_cache[location]

# Function to process a single row
def date_extraction(row):
    evidence = get_evidence_from_wikipedia(row['location'], row['description'])
    return evidence['date']

df["Haunted Places Date"] = df.apply(date_extraction, axis=1)

# Save updated data
#output_path = '../data/haunted_places.tsv'
#df.to_csv(output_path, index=False)
#logging.info(f"✅ New file saved with Wikipedia enrichment: {output_path}")

2025-03-13 14:47:23,177 - INFO - Wikipedia: language=en, user_agent: DSCI550_HauntedProject/1.0 (https://github.com/maggie-changg/dsci550-assignment1; changmag@usc.edu) (Wikipedia-API/0.8.1; https://github.com/martin-majlis/Wikipedia-API/), extract_format=1
2025-03-13 14:47:23,195 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Ada Cemetery&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2025-03-13 14:47:23,555 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=North Adams Rd.&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2025-03-13 14:47:23,725 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Ghost Trestle&inprop=protectio

In [22]:
df['Haunted Places Date']

0        2025/03/03
1        2025/03/01
2        2025/01/01
3        1919/03/13
4        1835/03/13
            ...    
10987    2025/01/01
10988    2025/05/13
10989    2025/03/18
10990    2025/01/01
10991    2025/01/01
Name: Haunted Places Date, Length: 10992, dtype: object

In [25]:
# Total number of entries
total_entries = len(df)
fallback_date_count = (df['Haunted Places Date'] == '2025/01/01').sum()
hasDates_count = total_entries - fallback_date_count

# Percentage of fallback dates
fallback_percentage = (fallback_date_count / total_entries) * 100
hasDates_percentage = (hasDates_count / total_entries) * 100
                       

print(f"\nNumber of fallback dates (2025/01/01): {fallback_date_count}")
print(f"Number of Entries with dates: {hasDates_count}")
print(f"Percentage of total entries without dates : {fallback_percentage:.2f}%")
print(f"Percentage of total entries with dates: {hasDates_percentage:.2f}%")


Number of fallback dates (2025/01/01): 4969
Number of Entries with dates: 6023
Percentage of total entries without dates : 45.21%
Percentage of total entries with dates: 54.79%
